<a href="https://colab.research.google.com/github/adriantang23/cs505-machine-translation/blob/main/notebooks/ibm_model1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Add Datasets
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Loead Datasets
fr_path = '/content/drive/MyDrive/cs505_data/europarl-v7.fr-en.fr'
en_path = '/content/drive/MyDrive/cs505_data/europarl-v7.fr-en.en'

with open(fr_path, encoding='utf-8') as f:
    fr_lines = f.readlines()
with open(en_path, encoding='utf-8') as f:
    en_lines = f.readlines()

print(f"Total sentence pairs: {len(fr_lines):,}")
print(f"FR: {fr_lines[0].strip()}")
print(f"EN: {en_lines[0].strip()}")

Total sentence pairs: 2,007,723
FR: Reprise de la session
EN: Resumption of the session


In [3]:
# Tokenization and subsets
def tokenize(sentence):
    return sentence.lower().strip().split()

TRAIN_SIZE = 50000
TEST_SIZE = 2000

train_fr = [tokenize(s) for s in fr_lines[:TRAIN_SIZE]]
train_en = [tokenize(s) for s in en_lines[:TRAIN_SIZE]]

test_fr  = [tokenize(s) for s in fr_lines[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]]
test_en_raw = [s.lower().strip() for s in en_lines[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]]

print(f"Train pairs: {len(train_fr):,}")
print(f"Test  pairs: {len(test_fr):,}")
print(f"Sample FR tokens: {train_fr[0]}")
print(f"Sample EN tokens: {train_en[0]}")


Train pairs: 50,000
Test  pairs: 2,000
Sample FR tokens: ['reprise', 'de', 'la', 'session']
Sample EN tokens: ['resumption', 'of', 'the', 'session']


In [4]:
# Initialize transition table, train next cell
from collections import defaultdict

def transition_probs_init(fr_corpus, en_corpus):
  fr_to_en = defaultdict(set)
  for fr_sentence, en_sentence in zip(fr_corpus, en_corpus):
      for fr_word in fr_sentence:
          for en_word in en_sentence:
              fr_to_en[fr_word].add(en_word)

  t = defaultdict(float)
  for fr_word, en_words in fr_to_en.items():
      for en_word in en_words:
          t[(en_word, fr_word)] = 1.0 / len(en_words)


  return t

t = transition_probs_init(train_fr, train_en)
print(f"Translation table entries: {len(t):,}")

Translation table entries: 9,847,502


In [5]:
# EM training
iterations = 10

for i in range(iterations):
    count = defaultdict(float)
    total = defaultdict(float)

    # E for counts
    for en_sentence, fr_sentence in zip(train_en, train_fr):
        for en_word in en_sentence:
            sums = sum(t[(en_word,fr_word)] for fr_word in fr_sentence)
            if sums == 0:
                continue
            for fr_word in fr_sentence:
                delta = t[(en_word,fr_word)] / sums
                count[en_word,fr_word] += delta
                total[fr_word] += delta

    # M for probabilities
    for en_word, fr_word in count:
        if total[fr_word] == 0:
            continue
        t[(en_word,fr_word)] = count[en_word,fr_word] / total[fr_word]

    top_pairs = sorted(t.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"Iteration {i + 1}/{iterations}")

print("\nTraining done")
print("Top 20 translations:")
top_20 = sorted(t.items(), key=lambda x: x[1], reverse=True)[:20]
for (en, fr), prob in top_20:
    print(f"  {fr} -> {en}: {prob:.4f}")



Iteration 1/10
Iteration 2/10
Iteration 3/10
Iteration 4/10
Iteration 5/10
Iteration 6/10
Iteration 7/10
Iteration 8/10
Iteration 9/10
Iteration 10/10

Training done
Top 20 translations:
  courts -> .: 1.0000
  absolument. -> absolutely.: 1.0000
  "jeunesse -> youth: 1.0000
  (applaudissements) -> (applause): 0.9999
  (suite) -> (continuation): 0.9963
  régional -> regional: 0.9907
  commissaire, -> commissioner,: 0.9904
  environnementaux -> environmental: 0.9900
  sociale -> social: 0.9898
  internationales -> international: 0.9890
  chocolat -> chocolate: 0.9880
  nationaux -> national: 0.9867
  sociaux -> social: 0.9864
  internationale -> international: 0.9863
  deuxièmement, -> secondly,: 0.9863
  trois -> three: 0.9854
  nationales -> national: 0.9850
  français -> french: 0.9820
  international -> international: 0.9813
  environnementales -> environmental: 0.9812


In [6]:
# Translation Lookup
best_translation = {}
for (en_word, fr_word), prob in t.items():
    if fr_word not in best_translation or prob > best_translation[fr_word][1]:
        best_translation[fr_word] = (en_word, prob)

def translate(fr_sentence):
    return [best_translation[fr_word][0] if fr_word in best_translation else "<UNK>"
            for fr_word in fr_sentence]

print("Samples")
for i in range(5):
    fr = train_fr[i]
    en_ref = train_en[i]
    en_pred = translate(fr)
    print(f"\nFR:  {' '.join(fr)}")
    print(f"REF: {' '.join(en_ref)}")
    print(f"IBM: {' '.join(en_pred)}")


Samples

FR:  reprise de la session
REF: resumption of the session
IBM: resumed of the session

FR:  je déclare reprise la session du parlement européen qui avait été interrompue le vendredi 17 décembre dernier et je vous renouvelle tous mes vux en espérant que vous avez passé de bonnes vacances.
REF: i declare resumed the session of the european parliament adjourned on friday 17 december 1999, and i would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.
IBM: i the resumed the session the parliament european which was been adjourned the friday 17 december last and i you again all my has in hope that you have past of good happy

FR:  comme vous avez pu le constater, le grand "bogue de l'an 2000" ne s'est pas produit. en revanche, les citoyens d'un certain nombre de nos pays ont été victimes de catastrophes naturelles qui ont vraiment été terribles.
REF: although, as you will have seen, the dreaded 'millennium bug' failed to materialise

In [7]:
# Save translation table to drive - no need to retrain
import pickle

save_path = '/content/drive/MyDrive/cs505_data/ibm_model1_t.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(dict(t), f)
print("Translation table saved to Drive")

Translation table saved to Drive


In [8]:
print(f"Test sentences: {len(test_fr):,}")
print(f"Sample FR: {' '.join(test_fr[0])}")
print(f"Sample EN: {test_en_raw[0]}")


Test sentences: 2,000
Sample FR: le nombre de cas augmente. c'est bien la tendance qui pose un problème ici, plus encore que l'importance de l'épidémie sur le plan quantitatif.
Sample EN: it is this tendency which presents a problem right now, much more than the scale of the epidemic in quantitative terms.


In [9]:
# sacrebleu for BLEU score
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.0 MB/s eta 0:00:00


In [10]:
import sacrebleu

translations = [' '.join(translate(s)) for s in test_fr]
bleu = sacrebleu.corpus_bleu(translations, [test_en_raw])
print(f"IBM Model 1 BLEU score (Europarl in-domain): {bleu.score:.2f}")

# Qualitative sample
print("\nSample translations:")
for i in range(5):
    print(f"\nFR:  {' '.join(test_fr[i])}")
    print(f"REF: {test_en_raw[i]}")
    print(f"IBM: {' '.join(translate(test_fr[i]))}")

IBM Model 1 BLEU score (Europarl in-domain): 10.98

Sample translations:

FR:  le nombre de cas augmente. c'est bien la tendance qui pose un problème ici, plus encore que l'importance de l'épidémie sur le plan quantitatif.
REF: it is this tendency which presents a problem right now, much more than the scale of the epidemic in quantitative terms.
IBM: the number of case increased is of the trend which is a problem here more still that importance of epidemic on the plan <UNK>

FR:  en effet, à l'heure actuelle, on ne sait tout simplement pas l'expliquer, d'où l'ouverture d'un débat sur l'existence possible de voies de transmission de la maladie que nous ignorerions encore.
REF: the truth is, at the present time, there is quite simply no adequate explanation for it, hence the debate which has been initiated on the possible existence of alternative disease transmission routes of which we are, as yet, unaware.
IBM: in indeed, to time present we not knows all simply not <UNK> hence opening a

In [11]:
# # Test
# x = 10
# for i in range(x):
#   print(f"{i+1}. {fr_lines[i].strip()}")
#   print(f"{i+1}. {en_lines[i].strip()}")
#   print('\n')

# # Check common pairs
# test_pairs = [
#     ("the", "le"),
#     ("the", "la"),
#     ("of", "de"),
#     ("is", "est"),
#     ("and", "et"),
#     ("in", "dans"),
#     ("that", "que"),
#     ("it", "il"),
# ]

# print("After EM:")
# for en, fr in test_pairs:
#     prob = t[(en, fr)]
#     print(f"  {fr} -> {en}: {prob:.4f}")